# 02 Residual Stream / Logit Lens / Activation Patching

## このノートブックの目的

Qwen3-4B の内部計算を 3 つの視点から観察する。

| 概念 | 概要 |
|------|------|
| **Residual Stream** | 各 Transformer layer の入出力となる hidden state のベクトル系列 |
| **Logit Lens** | 各 layer 後の hidden state を lm_head に通して「その時点でのモデルの予測」を読む |
| **Activation Patching** | clean run の hidden state を corrupt run に注入し、各 layer の因果的寄与を測る |

### 実験設定

| | プロンプト | 期待する次トークン |
|---|---|---|
| **clean** | `"The capital of Japan is"` | ` Tokyo` |
| **corrupt** | `"The capital of France is"` | ` Paris` |

clean run の hidden state を layer k で corrupt run に注入したとき、出力がどう変化するかを計測する。


## 1. 環境セットアップ

In [ ]:
%matplotlib inline
import json
from pathlib import Path
import torch
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODEL_ID = "Qwen/Qwen3-4B"

# Jupyter はこのノートのあるフォルダから起動してください
outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)
print(f"model_id : {MODEL_ID}")
print(f"outputs  : {outputs_dir.name}/")

# デバイス選択
if torch.cuda.is_available():
    device = "cuda"
    torch_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32
print(f"device   : {device}")
print(f"dtype    : {torch_dtype}")


## 2. モデルとトークナイザーの読み込み

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

# トークナイザーの読み込み
print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"  vocab_size = {tokenizer.vocab_size}")

# モデルの読み込み
print("Loading model ...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    attn_implementation="eager",
)
model.to(device)  # GPU/MPS/CPU に転送  # pyright: ignore[reportArgumentType]
model.eval() # 推論専用の動作に設定（Dropout オフなど）
torch.set_grad_enabled(False)  # 計算グラフ構築オフ
print(f"  device     : {next(model.parameters()).device}")
print(f"  dtype      : {next(model.parameters()).dtype}")

K: int = model.config.num_hidden_layers
hidden_size: int = model.config.hidden_size
print(f"  layers K   : {K}")
print(f"  hidden_size: {hidden_size}")


### ヘルパー関数 (tokenizer使用)

In [ ]:
# トークンテーブルを表示
def show_token_table(text: str) -> pd.DataFrame:
    """テキストをトークナイズして位置・ID・piece の表を返す。"""

    # add_special_tokens=False: BOS などを付加せず、テキスト本体だけをトークン化
    ids = tokenizer.encode(text, add_special_tokens=False)

    rows = []
    for pos, tid in enumerate(ids):
        # convert_ids_to_tokens: トークナイザー内部の raw 表現（Ġ 付きなど）
        piece = tokenizer.convert_ids_to_tokens(tid)
        # decode: token ID を人間が読めるテキストに変換
        decoded = tokenizer.decode([tid])
        rows.append({
            "pos":      pos,
            "token_id": tid,
            "piece":    piece,   
            "decoded":  repr(decoded), # 'capital' のように表示
        })
    return pd.DataFrame(rows).set_index("pos")

# 確率上位 k トークンの表を返す
def topk_table(logits: torch.Tensor, k: int = 10) -> pd.DataFrame:
    """logits [vocab] から確率上位 k トークンの DataFrame を返す。"""

    # logits をソフトマックスで確率に変換（float32 で計算して精度を確保）
    probs = torch.softmax(logits.float(), dim=-1)

    # 確率上位 k 個の値とインデックスを取得
    top_vals, top_ids = torch.topk(probs, k)

    rows = []
    for rank, (tid, prob) in enumerate(zip(top_ids.tolist(), top_vals.tolist()), start=1):
        decoded = tokenizer.decode([tid])   # token ID → テキスト
        rows.append({
            "rank":     rank,
            "token_id": tid,
            "decoded":  repr(decoded),    # 先頭スペースなどを repr で可視化
            "logit":    logits[tid].item(),
            "prob":     prob,
        })
    return pd.DataFrame(rows).set_index("rank")

## 3. トークンテーブル

In [ ]:
# プロンプトと答え
CLEAN_PROMPT   = "The capital of Japan is"
CORRUPT_PROMPT = "The capital of France is"
CLEAN_ANSWER   = " Tokyo"
CORRUPT_ANSWER = " Paris"

clean_ans_ids   = tokenizer.encode(CLEAN_ANSWER,   add_special_tokens=False)
corrupt_ans_ids = tokenizer.encode(CORRUPT_ANSWER, add_special_tokens=False)

CLEAN_ANS_ID   = clean_ans_ids[0]
CORRUPT_ANS_ID = corrupt_ans_ids[0]
print(f"CLEAN_ANSWER   {CLEAN_ANSWER!r} -> id = {CLEAN_ANS_ID}")
print(f"CORRUPT_ANSWER {CORRUPT_ANSWER!r} -> id = {CORRUPT_ANS_ID}")
if len(clean_ans_ids) != 1 or len(corrupt_ans_ids) != 1:
    print("[warning] answer string is not a single token")


In [ ]:
df_clean_tok = show_token_table(CLEAN_PROMPT)
display(df_clean_tok.style.set_caption("clean prompt token table"))
clean_pos = len(df_clean_tok) - 1

df_corrupt_tok = show_token_table(CORRUPT_PROMPT)
display(df_corrupt_tok.style.set_caption("corrupt prompt token table"))
corrupt_pos = len(df_corrupt_tok) - 1

## 4. モデルの最終出力：next-token distribution

In [ ]:
# clean run
clean_inputs = tokenizer(CLEAN_PROMPT, return_tensors="pt").to(device)
clean_outputs = model(
    **clean_inputs,
    output_hidden_states=True,   # 全 layer の hidden state を返す（logit lens に必要）
    output_attentions=False,     # attention weights は不要（メモリ節約）
    use_cache=False,             # KV cache 無効（内部状態観察時は不要）
)

# 残差ストリーム　（全層，トークン列の全体）
# hidden_states: K+1 個のテンソルのタプル, 各テンソルの shape = [1, seq_len, hidden_size]
#   hs[0]   = embed_tokens の出力
#   hs[k]   = layers[k-1] の出力, k=1,2,...,K-1
#   hs[K]   = layers[K-1]の出力に，final normを適用したもの
clean_hs = clean_outputs.hidden_states

# 次トークンの予測分布の logit 値　（トークン列の全体）
# logits: shape [1, seq_len, vocab_size]　（ここではテキスト一つだからバッチサイズは1）
clean_logits = clean_outputs.logits[0, clean_pos, :].float() # clean_posの logit ベクトル
#   shape [vocab_size] の1次元テンソルになる
clean_logits.shape

In [ ]:
# softmax で logit → 確率に変換（全vocabの確率の和 = 1）
#   dim=-1: 最後の次元（vocab_size 方向）に沿って softmax を適用
#   clean_logits は 1次元テンソルなので dim=0 と dim=-1 は等価（多次元では異なる）
clean_probs  = torch.softmax(clean_logits, dim=-1)
print(f"clean run:")
print(f"  top-1   = {repr(tokenizer.decode([clean_logits.argmax().item()]))}")
print(f"  P({CLEAN_ANSWER.strip()}) = {clean_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P({CORRUPT_ANSWER.strip()}) = {clean_probs[CORRUPT_ANS_ID]:.4f}")

In [ ]:
# corrupt run
corrupt_inputs = tokenizer(CORRUPT_PROMPT, return_tensors="pt").to(device)
corrupt_outputs = model(
    **corrupt_inputs,
    output_hidden_states=True,
    output_attentions=False,
    use_cache=False,
)

corrupt_hs     = corrupt_outputs.hidden_states
corrupt_logits = corrupt_outputs.logits[0, corrupt_pos, :].float()
corrupt_probs  = torch.softmax(corrupt_logits, dim=-1)
print(f"corrupt run:")
print(f"  top-1   = {repr(tokenizer.decode([corrupt_logits.argmax().item()]))}")
print(f"  P({CLEAN_ANSWER.strip()}) = {corrupt_probs[CLEAN_ANS_ID]:.4f}")
print(f"  P({CORRUPT_ANSWER.strip()}) = {corrupt_probs[CORRUPT_ANS_ID]:.4f}")


In [ ]:
display(topk_table(clean_logits, k=10).style.set_caption("clean top-10"))
display(topk_table(corrupt_logits, k=10).style.set_caption("corrupt top-10"))

Tokyo/Parisのかわりにin, located, 下線, Kyotoなど，英文として有り得そうな単語も含まれている


## 5. 入口と出口：embedding / unembedding / softmax

Section 4 では、clean prompt と corrupt prompt に対して、モデルが最終的に出力する next-token distribution を確認しました。

この節では、その分布がどのように作られるかを、モデルの入口と出口に分けて確認します。  
最初に、後で使う記号を準備しておきます。

- 語彙集合 (vocabulary) を $\mathcal V$ とし、語彙数を $M = |\mathcal V|$ とします。
- hidden state の次元（hidden dimension）を $d$ とします。
- Qwen3-4B では $M = 151936$, $d = 2560$ です。
- 長さ$T$の 入力 token 列を $x_0, x_1, \dots, x_{T-1}$ とします。
- $t = 0, 1, \ldots, T-1$ は、トークン列中の位置 (position) を表す添字です。
- 各 $x_t$ は token ID で、$x_t \in \mathcal V$ と考えます。

言語モデルの入力は文字列そのものではなく、この token ID の列です。  
この節の流れは次の通りです。

- 入口：token ID $x_t$ から embedding vector $h_t^{(0)}$ を作る。
- 中間：Transformer blocks が hidden state $h_t^{(0)}, h_t^{(1)}, \dots, h_t^{(K)}$ を順に更新していく（residual stream）。
- 出口：final hidden state $h_t^{(K)}$ を `lm_head` / unembedding に通して logits $z_t$ にする。
- 確率化：logits $z_t$ を softmax にかけて next-token distribution $p_t$ にする。


### 入口：token ID から embedding へ

各 token ID $i \in \mathcal V$ に、対応する input embedding vector

$$
v_i^{\mathrm{in}} \in \mathbb{R}^d
$$

を割り当てます。

これらを語彙全体について縦に並べた行列を、input embedding matrix $W_E$ と呼びます。

$$
W_E =
\begin{pmatrix}
(v_0^{\mathrm{in}})^\top \\
(v_1^{\mathrm{in}})^\top \\
\vdots \\
(v_{M-1}^{\mathrm{in}})^\top
\end{pmatrix}
\in \mathbb{R}^{M \times d}
$$

位置 $t$ における最初の hidden state $h_t^{(0)}$ は、token ID $x_t$ に対応する input embedding vector そのものとして定義します。

$$
h_t^{(0)} = v_{x_t}^{\mathrm{in}} \in \mathbb{R}^d
$$

つまり、入口側では、$W_E$ の中から token ID $x_t$ に対応する行ベクトル $v_{x_t}^{\mathrm{in}}$ を参照し、それを residual stream の初期値 $h_t^{(0)}$ にします。実装上は、この参照操作は embedding lookup と呼ばれます。

PyTorch では、この lookup は

```python
model.model.embed_tokens(input_ids)
```

として実行されます。Hugging Face Transformers の `outputs.hidden_states[0]` は、この input embedding の出力に対応します。


In [ ]:
# 入口: token ID -> embedding
input_ids = clean_inputs["input_ids"]
emb = model.model.embed_tokens(input_ids)

print("input_ids.shape       =", tuple(input_ids.shape))
print("embedding.shape       =", tuple(emb.shape))
print("hidden_states[0].shape =", tuple(clean_hs[0].shape))


Python オブジェクトと数式の対応：

| Python | 意味 | shape |
|---|---|---|
| `input_ids` | token ID 列 $x_0, x_1, \dots, x_{T-1}$ | $[1, T]$ |
| `emb` | $W_E$ から lookup した input embedding vector $v_{x_t}^{\mathrm{in}}$ を $t = 0, 1, \dots, T-1$ について並べたテンソル | $[1, T, d]$ |
| `clean_hs[0]` | residual stream の初期値 $h_t^{(0)}$ を $t = 0, 1, \dots, T-1$ について並べたテンソル（`outputs.hidden_states[0]`） | $[1, T, d]$ |

ここで $T$ は input token 数、$d = 2560$ は hidden dimension、バッチ次元 $[1, \cdot]$ はサンプル数 1 です。

定義上 $h_t^{(0)} = v_{x_t}^{\mathrm{in}}$ なので、`emb` と `clean_hs[0]` は同じ値になります。続く code cell でも `print` した値が一致することを確認できます。


ちょっと実際に見てみます

In [ ]:
# トークン列
print(input_ids)

# トークン列の embedding
print(emb)

# 入力層の hidden state
print(clean_hs[0])

# sanity check: 2 つのテンソルの成分ごとの最大差
diff = (emb.float() - clean_hs[0].float()).abs().max().item()
print(f"成分の最大差 = {diff:.4e}")

### 出口：hidden state から logits へ

入口で得られた $h_t^{(0)}$ は、Transformer blocks を通って順に更新されていきます。

$$
h_t^{(0)}
\;\longrightarrow\;
h_t^{(1)}
\;\longrightarrow\;
\cdots
\;\longrightarrow\;
h_t^{(K)}
$$

ここで $K$ は Transformer block の数で、Qwen3-4B では

$$
K = 36
$$

です。

この notebook では、$h_t^{(K)}$ を「最後の Transformer block を通り、さらに final RMSNorm を通った後の hidden state」として扱います。これは `lm_head` に入力される直前の hidden state です。

次に、出口側でも、各 token ID $i \in \mathcal V$ に、対応する unembedding vector

$$
v_i^{\mathrm{out}} \in \mathbb{R}^d
$$

を割り当てます。これらを語彙全体について縦に並べた行列を、unembedding matrix $W_U$ と呼びます。

$$
W_U =
\begin{pmatrix}
(v_0^{\mathrm{out}})^\top \\
(v_1^{\mathrm{out}})^\top \\
\vdots \\
(v_{M-1}^{\mathrm{out}})^\top
\end{pmatrix}
\in \mathbb{R}^{M \times d}
$$

position $t$ における token $i$ の logit は、$h_t^{(K)}$ と $v_i^{\mathrm{out}}$ の内積として定義します。

$$
z_{t,i}
=
(v_i^{\mathrm{out}})^\top h_t^{(K)} \in \mathbb{R}
$$

これは、next token、すなわち position $t+1$ において token $i$ の出現しやすさ、をあらわす score です（まだ確率ではありません）。

vocabulary 全体に対する logits ベクトル $z_t = (z_{t,0},\ldots,z_{t,M-1})^\top$  は、$W_U$ を使って次のように書けます。

$$
z_t
=
W_U \, h_t^{(K)}
\in \mathbb{R}^M
$$

`lm_head` がこの $W_U$ を持つ線形層に対応します（`lm_head.weight` が $W_U$）。  

要するに、最終 hidden state $h_t^{(K)}$ を `lm_head` に通すことで logits $z_t$ になります。これは、vocabulary 全体に対する score です。logits はまだ確率ではない点に注意してください。


In [ ]:
# 出口: final hidden state -> lm_head -> logits
h_final = clean_hs[-1][:, clean_pos, :]      # [1, hidden_size]
logits_manual = model.lm_head(h_final)       # [1, vocab_size]

print("h_final.shape         =", tuple(h_final.shape))
print("lm_head.weight.shape  =", tuple(model.lm_head.weight.shape))
print("logits_manual.shape   =", tuple(logits_manual.shape))
print("clean_logits.shape    =", tuple(clean_logits.shape))


Python オブジェクトと数式の対応：

| Python | 意味 | shape |
|---|---|---|
| `clean_hs[-1]` | residual stream の最終値 $h_t^{(K)}$ を $t = 0, 1, \dots, T-1$ について並べたテンソル（`outputs.hidden_states[K]`、final RMSNorm 適用後） | $[1, T, d]$ |
| `h_final` | 上から $t=$`clean_pos` の位置を取り出した $h_{\text{clean\_pos}}^{(K)}$ | $[1, d]$ |
| `model.lm_head.weight` | unembedding matrix $W_U$ | $[M, d]$ |
| `logits_manual` | $z_{\text{clean\_pos}} = W_U \, h_{\text{clean\_pos}}^{(K)}$ を手計算したもの（バッチ次元付き） | $[1, M]$ |
| `clean_logits` | 同じ位置の logits を forward pass の `clean_outputs.logits` からスライスして取り出したもの | $[M]$ |

ここで $K = 36$、$M = 151936$、$d = 2560$ です。  
`logits_manual[0]` と `clean_logits` は同じ $z_{\text{clean\_pos}}$ を表すため、差分はほぼ 0 になります。


ちょっと実際に見てみます


In [ ]:
# clean_pos における final hidden state
print(h_final)

# 手計算した logits (lm_head による)
print(logits_manual)

# forward pass からの logits
print(clean_logits)

# sanity check: 2 つのベクトルの成分ごとの最大差
diff = (logits_manual[0].float() - clean_logits).abs().max().item()
print(f"成分の最大差 = {diff:.4e}")

### softmax：logits から確率分布へ

logits $z_t$ は vocabulary 全体の score です。  
これを確率分布に変換するには softmax を使います。

$$
p_{t,i}
=
\frac{\exp(z_{t,i})}{\sum_{j \in \mathcal V}\exp(z_{t,j})},\quad i \in \mathcal V
$$

これは語彙集合 $\mathcal V$ 全体に対する確率分布になっています。つまり、$p_{t,i}>0$、

$$
\sum_{i\in \mathcal V} p_{t,i} = 1
$$

です。ベクトルでは、$p_t = (p_{t,0},\ldots, p_{t,M-1})^\top$ として、

$$
p_t = \mathrm{softmax}(z_t)
$$

と書きます。softmax 後の $p_{t,i}$ は、位置 $t$ の次に token $i$ が来る確率として解釈できます。

つまり、モデルは `" Tokyo"` と `" Paris"` の二択だけをしているわけではなく、語彙全体に対する分布を出していることに注意してください。


In [ ]:
# softmax: logits -> probability distribution
probs_manual = torch.softmax(logits_manual[0].float(), dim=-1)

print("probs_manual.shape =", tuple(probs_manual.shape))
print(f"sum(probs_manual) = {probs_manual.sum().item():.6f}")

display(topk_table(logits_manual[0].float(), k=10).style.set_caption("manual lm_head top-10"))


### embedding と unembedding の重み共有

入口で使った input embedding vector $v_i^{\mathrm{in}}$ と、出口で使う unembedding vector $v_i^{\mathrm{out}}$ は、概念上は役割が異なります。

- 入口では、token ID $x_t$ に対応する $v_{x_t}^{\mathrm{in}}$ を参照し、$h_t^{(0)}$ を作ります。
- 出口では、hidden state $h_t^{(K)}$ と各 $v_i^{\mathrm{out}}$ の内積を計算し、token $i$ の logit $z_{t,i}$ を作ります。

Qwen3-4B では、設定上 `tie_word_embeddings=True` であり、実装上も `embed_tokens.weight` と `lm_head.weight` は同じ tensor を共有しています。  
そのため、Qwen3-4B においては、各 token ID $i$ について

$$
v_i^{\mathrm{in}} = v_i^{\mathrm{out}}
$$

が成り立ち、これらは同じ tensor の同じ行に対応します。

ただし、これは入口と出口が数学的な逆変換であるという意味ではありません。  
同じ token vector table を、入口では token ID から hidden vector を作り始めるために使い、出口では hidden state から token ごとの score を読むために使っている、と考えるのが安全です。


In [ ]:
# embedding matrix and unembedding matrix
W_E = model.model.embed_tokens.weight
W_U = model.lm_head.weight

print("W_E.shape =", tuple(W_E.shape))
print("W_U.shape =", tuple(W_U.shape))
print("tie_word_embeddings =", model.config.tie_word_embeddings)
print("same tensor =", W_E.data_ptr() == W_U.data_ptr())

if W_E.data_ptr() == W_U.data_ptr():
    print("max |W_E - W_U| = 0.0000e+00  (same tensor)")
else:
    diff_WE_WU = (W_E.float() - W_U.float()).abs().max().item()
    print(f"max |W_E - W_U| = {diff_WE_WU:.4e}")


ここまでで、モデルの入口と出口を確認しました。

- 語彙集合を $\mathcal V$、語彙数を $M$、hidden dimension を $d$ とした。
- 入口では、token ID $x_t$ に対応する input embedding vector $v_{x_t}^{\mathrm{in}}$ が $h_t^{(0)}$ になる。
- 中間では、Transformer blocks が $h_t^{(0)}, h_t^{(1)}, \dots, h_t^{(K)}$ を順に更新する（residual stream）。
- 出口では、final hidden state $h_t^{(K)}$ が `lm_head` / unembedding によって logits $z_t \in \mathbb{R}^M$ になる。
- softmax により logits $z_t$ は next-token distribution $p_t$ になる。
- Qwen3-4B では input embedding と unembedding の重みは共有されているが、入口と出口の役割は異なる。

次の Section 6 では、この中間にある `hidden_states[k]` と residual stream の対応を確認し、その途中層を `lm_head` で読む logit lens に進みます。


## 6. residual stream と hidden_states の対応

前節では、入口の embedding と出口の `lm_head` / softmax を確認しました。  
ここからは、その中間を流れる hidden state の系列 $h_t^{(0)}, h_t^{(1)}, \dots, h_t^{(K)}$ を **residual stream** として見ていきます。

Hugging Face Transformers の `outputs.hidden_states` は長さ $K + 1$ のタプルで、その第 $k$ 要素 `hidden_states[k]`（このノートでは `clean_hs[k]`）が、$h_t^{(k)}$ を $t = 0, 1, \dots, T-1$ について並べた `[1, T, d]` テンソルに対応します。


In [ ]:
print(f"K = {K}")
print(f"len(clean_hs) = {len(clean_hs)}  (= K+1 = {K}+1)")
print(f"hs[0].shape   = {tuple(clean_hs[0].shape)}  <- embed_tokens 出力")
print(f"hs[1].shape   = {tuple(clean_hs[1].shape)}  <- layer 0 出力")
print(f"hs[K].shape   = {tuple(clean_hs[K].shape)}  <- norm 後 (k=K)")
print()
print("Residual stream のインデックス対応:")
print("  hs[0]   = embed_tokens(input_ids)")
print("  hs[k]   = layers[k-1] の出力  (1 <= k <= K-1)")
print(f"  hs[{K}]  = model.model.norm(layers[{K-1}] の出力)  ← lm_head の直前")

### logit lens の読み出し関数

本来の目的である、言語モデルによる推論（テキストの生成）では、logit lens は不要です。logit lens は、言語モデルの内部で情報がどのように処理されているかを調べるための**解析ツール**です。途中層の hidden state $h_t^{(k)}$ を見ても、それが何を表すのかわかりませんが、$h_t^{(k)}$ を出口と同じ仕組みで vocabulary 全体の score に変換することで、「その時点でのモデルの予測」として解釈できるようになります。これを **logit lens** と呼びます。

具体的には、`lm_head` の直前に置かれている final RMSNorm $\mathrm{Norm}(\cdot)$（PyTorch では `model.model.norm` に対応）と unembedding $W_U$ を組み合わせて、$h_t^{(k)}$ を vocabulary 全体の logits に変換します。

$$
z_t^{(k)}
= W_U \, \mathrm{Norm}(h_t^{(k)})
\in \mathbb{R}^M
\quad (0 \le k \le K-1)
$$

最終層 $k = K$ については、Hugging Face Transformers の `outputs.hidden_states[K]` (= `clean_hs[-1]`) がすでに $\mathrm{Norm}$ 適用後のテンソルなので、追加の norm は不要で、

$$
z_t^{(K)} = W_U \, h_t^{(K)} = z_t
$$

となり、forward pass の最終 logits $z_t$ と一致します（次セルの sanity check で確認します）。

なお、ここでの $k$ の場合分け（$k < K$ で Norm 必要、$k = K$ で不要）は、前セルで述べた `hs[k]` の由来による分類（embed 出力 / layer 出力 / norm 後）とは異なり、`lm_head` 直前の Norm 適用の有無だけで決まります。

各層の「途中段階の予測分布」は、softmax をかけて

$$
p_t^{(k)} = \mathrm{softmax}(z_t^{(k)}) \in \mathbb{R}^M
$$

として得ます。

次に定義する `logit_lens(hidden_states, k, pos)` は、この $z_t^{(k)}$ を $k < K$ と $k = K$ で場合分けして計算します。続く `logit_lens_sweep(hidden_states, pos)` は、全ての $k = 0, 1, \dots, K$ について $z_t^{(k)}$ と $p_t^{(k)}$ を計算し、top-1 token などとともに DataFrame にまとめます。なお、いずれの関数も `model` と `tokenizer` はこの notebook 共通のものを外スコープから直接参照します。


In [ ]:
# Logit Lens の計算
def logit_lens(hidden_states, k: int, pos: int) -> torch.Tensor:
    """
    残差ストリームの layer k を「その時点でのモデルの予測」として読むために，
    その時点での hidden state を lm_head に通した logits を返す。
    k = K だけ処理が異なることに注意する。

    hidden_states のインデックス対応:
      hs[0]   = embed_tokens の出力
      hs[k]   = layers[k-1] の出力  （1 <= k <= K-1）
      hs[K]   = model.model.norm 後  （lm_head の直前）

    k < K の場合は norm を通してから lm_head に入力する。
    k = K の場合は hs[K] が既に norm 後なので直接 lm_head に入力する。

    model と tokenizer は外スコープのものを参照する。
    """
    K = len(hidden_states) - 1  # Qwen3-4B では K=36
    hs = hidden_states[k]       # shape: [1, seq, hidden_size]

    if k < K:
        # pos:pos+1 でスライスして [1, 1, hidden_size] にしてから norm を適用
        normed = model.model.norm(hs[:, pos:pos+1, :])  # [1, 1, hidden_size]
        logits = model.lm_head(normed)[:, 0, :]          # [1, vocab]
    else:
        # k == K: hs[K] は既に norm 済み。pos だけ取り出して lm_head へ
        normed = hs[:, pos, :]         # [1, hidden_size]
        logits = model.lm_head(normed) # [1, vocab]

    return logits[0]  # [vocab]


# 全 layer の logit lens を sweep して DataFrame を返す
def logit_lens_sweep(hidden_states, pos: int) -> pd.DataFrame:
    """hidden_states の各層を logit lens で読み、確率等をまとめた DataFrame を返す。"""
    K_loc = len(hidden_states) - 1
    rows = []
    for k in range(K_loc + 1):
        ll_logits    = logit_lens(hidden_states, k, pos)
        ll_probs     = torch.softmax(ll_logits.float(), dim=-1)
        top1_id      = int(ll_logits.argmax().item())
        top1_decoded = tokenizer.decode([top1_id])
        site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K_loc else "norm")
        rows.append({
            "k":            k,
            "site":         site,
            "top1_decoded": repr(top1_decoded),
            "p_top1":       ll_probs[top1_id].item(),
            "p_clean":      ll_probs[CLEAN_ANS_ID].item(),
            "p_corrupt":    ll_probs[CORRUPT_ANS_ID].item(),
        })
    return pd.DataFrame(rows)


### logit lens の sanity check：k=K は最終出力と一致する

Section 5 で見たとおり、$z_t^{(K)} = W_U \, h_t^{(K)} = z_t$ が成り立つはずです。`logit_lens(clean_hs, K, clean_pos)` の出力が forward pass の最終 logits `clean_logits` と（誤差の範囲で）一致することを確認します。


In [ ]:
# hs[K] (norm 後) を lm_head に通した結果と model output logits を比較
ll_logits_K = logit_lens(clean_hs, K, clean_pos)
display(topk_table(ll_logits_K, k=5).style.set_caption("logit_lens(k=K)"))
display(topk_table(clean_logits, k=5).style.set_caption("model output logits"))

diff = (ll_logits_K - clean_logits).abs().max().item()
print(f"logit_lens(k=K) vs model logits: max abs diff = {diff:.6f}")
print("  diff ≈ 0 なら logit_lens の実装が正しい")

## 7. Logit Lens — clean run

In [ ]:
# clean run
df_ll_clean = logit_lens_sweep(clean_hs, clean_pos)
display(df_ll_clean.set_index("k").style.set_caption("Logit Lens — clean run").format({"p_top1": "{:.4f}", "p_clean": "{:.4f}", "p_corrupt": "{:.4f}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_ll_clean["k"], df_ll_clean["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title(f"Logit Lens — clean run  ({CLEAN_PROMPT})")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_clean.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


## 8. Logit Lens 比較 — clean vs corrupt

In [ ]:
# corrupt run
df_ll_corrupt = logit_lens_sweep(corrupt_hs, corrupt_pos)
display(df_ll_corrupt.set_index("k").style.set_caption("Logit Lens — corrupt run").format({"p_top1": "{:.4f}", "p_clean": "{:.4f}", "p_corrupt": "{:.4f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax = axes[0]
ax.plot(df_ll_clean["k"], df_ll_clean["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_clean["k"], df_ll_clean["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — clean  ({CLEAN_PROMPT})")
ax.set_xlabel("layer k")
ax.set_ylabel("probability")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_clean"], label=f"P({CLEAN_ANSWER})", marker="o", markersize=3)
ax.plot(df_ll_corrupt["k"], df_ll_corrupt["p_corrupt"], label=f"P({CORRUPT_ANSWER})", marker="s", markersize=3, linestyle="--")
ax.set_title(f"Logit Lens — corrupt  ({CORRUPT_PROMPT})")
ax.set_xlabel("layer k")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_comparison.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


### logit-difference metric で見る logit lens

確率 plot は後半層で 0/1 に飽和するため、前半・中盤の変化が見えにくい場合がある。
`metric_k = logit_k(" Tokyo") - logit_k(" Paris")` を clean/corrupt で並べると、変化がより見やすくなる。


In [ ]:
# ２単語の出現しやすさの差を計算
def metric(logits: torch.Tensor, clean_id: int, corrupt_id: int) -> float:
    """
    logit(clean_answer) - logit(corrupt_answer) を返す。

    この値が大きいほど clean 側の答えが有利な状態。
    recovery の計算に使う基準指標。
    """
    return (logits[clean_id] - logits[corrupt_id]).item()

In [ ]:
# logit-difference metric plot (clean / corrupt)
ll_metric_clean   = [
    metric(logit_lens(clean_hs,   k, clean_pos), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]
ll_metric_corrupt = [
    metric(logit_lens(corrupt_hs, k, corrupt_pos), CLEAN_ANS_ID, CORRUPT_ANS_ID)
    for k in range(K + 1)
]

ks = list(range(K + 1))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks, ll_metric_clean,   label=f"clean  ({CLEAN_PROMPT})",  marker="o", markersize=3)
ax.plot(ks, ll_metric_corrupt, label=f"corrupt ({CORRUPT_PROMPT})", marker="s", markersize=3, linestyle="--")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("layer k  (0=embed, K=norm)")
ax.set_ylabel(f"logit({CLEAN_ANSWER}) - logit({CORRUPT_ANSWER})")
ax.set_title("Logit Lens — metric_k  (logit difference)")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = outputs_dir / "nb02_logit_lens_metric.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")


## 9. Activation Patching の概念と実装

### 概念

Activation Patching（残差ストリームパッチ）の手順：

1. **clean run** を実行 → 各 layer の hidden state `clean_hs[k]` を記録
2. **corrupt run** に hook を設置：layer $k$ の出力のうち `corrupt_pos` 位置だけを、`clean_hs[k][0, clean_pos, :]` で上書き
3. corrupt run を再実行 → 最終 logits を記録
4. **回復率 (recovery)** を計算

直感的には、「clean run のある層の活性を corrupt run に注入したとき、最終出力がどれだけ clean 側に戻るか」を層ごとに測ります。  
この回復の度合いを表す指標として **$\text{recovery}(k)$** を後で定義しますが、先取りして直感を述べると次のようになります。

- $\text{recovery}(k) \approx 0$ → layer $k$ のパッチは効果なし
- $\text{recovery}(k) \approx 1$ → layer $k$ のパッチで clean run に完全回復


### activation patching を設定するコード

以下に実装の核となる `run_patch(k)` を示す。詳細は後の数式定式化セクションで補足する。

In [ ]:
def run_patch(k: int) -> torch.Tensor:
    """
    corrupt run の層 k 出力 (corrupt_hs[k][0, corrupt_pos, :]) を
    clean  run の層 k 出力 (clean_hs[k][0,  clean_pos, :])  で置換して
    最終 logits [vocab] を返す。

    Transformers 5.x: Qwen3DecoderLayer.forward() は tensor を直接返す（tuple でない）。
    hook は out.clone() してから置換し、tensor を return する。
    """
    # clean run の層 k における clean_pos の残差ストリームを取り出す
    patch_vec = clean_hs[k][0, clean_pos, :].to(device)

    # patch する対象モジュールを k に応じて選ぶ
    #   k == 0       : embed_tokens（埋め込み層の出力）
    #   1 <= k <= K-1: layers[k-1]（第 k-1 デコーダ層の出力）
    #   k == K       : norm（最終 LayerNorm の出力）
    if k == 0:
        target_module = model.model.embed_tokens
    elif k < K:
        target_module = model.model.layers[k - 1]
    else:
        target_module = model.model.norm

    # forward hook: target_module の出力が計算されるたびに呼ばれる
    #   out[0, corrupt_pos, :] だけを patch_vec で上書きし、残りはそのまま返す
    def hook(module, inp, out):
        out = out.clone()                        # 元テンソルを破壊しないようコピー
        out[0, corrupt_pos, :] = patch_vec       # 指定位置だけ置換
        return out                               # 後続の層はこの値を受け取る

    # hook を登録して corrupt run を再実行し、終了後に必ず hook を解除する
    handle = target_module.register_forward_hook(hook)
    try:
        patched_out = model(
            **corrupt_inputs,
            output_hidden_states=False,
            output_attentions=False,
            use_cache=False,
        )
    finally:
        handle.remove()   # 例外が起きても hook が残り続けないように

    # corrupt_pos における最終 logits を返す
    return patched_out.logits[0, corrupt_pos, :].float()

### 数式での定式化

**記号の準備**

以降、$k$ は **パッチをかけた layer index**（sweep 軸、$\text{recovery}(k)$ の $k$）を表します。  
residual stream の自由変数（現在見ている layer index）は $k'$（$0 \le k' \le K$）と書きます。

clean prompt と corrupt prompt それぞれに対する forward pass の residual stream を区別します。

- $h_t^{(k'),\text{clean}}$ は、clean run の hidden state
- $h_t^{(k'),\text{corrupt}}$ は、corrupt run の hidden state
- 注目する位置は、clean run では `clean_pos`、corrupt run では `corrupt_pos`（いずれも last position）

**Patching の操作**

layer $k$ にパッチをかけた状態の residual stream を $h_t^{(k'),\text{patched}(k)}$ と書きます。

- $k'=0,\ldots,k-1$では、corrupt run で forward します。つまり、$$h_t^{(k'),\text{patched}(k)} = h_t^{(k'),\text{corrupt}}$$

- $k'=k$では、**位置 $t=$ `corrupt_pos` だけ** clean run の対応する活性で上書き
  $$
  h_{\text{corrupt\_pos}}^{(k),\text{patched}(k)} := h_{\text{clean\_pos}}^{(k),\text{clean}}
  $$
します。その他の位置では、通常通り forward します。

- その後の、$k'=k+1,\ldots,K$では、通常通り forward します。

要するに、$k'=k$ の $t=$ `corrupt_pos` だけ、clean run の `clean_pos` で上書きし、それ以外は、通常どおり forward して得られる residual stream として定義します。

**最終 logits**

Section 6 の規約 ($h_t^{(K)}$ は post-norm) に従い、$W_U$ を作用させて最終 logits を得ます。

$$
z^{\text{clean}}      = W_U \, h_{\text{clean\_pos}}^{(K),\text{clean}}, \quad
z^{\text{corrupt}}    = W_U \, h_{\text{corrupt\_pos}}^{(K),\text{corrupt}}, \quad
z^{\text{patched}(k)} = W_U \, h_{\text{corrupt\_pos}}^{(K),\text{patched}(k)}
$$

**Logit-difference metric**

clean 側の答え token（` Tokyo`、`CLEAN_ANS_ID`）と corrupt 側の答え token（` Paris`、`CORRUPT_ANS_ID`）の logit 差を metric とします。

$$
m(z) = z_{\text{CLEAN\_ANS\_ID}} - z_{\text{CORRUPT\_ANS\_ID}}
$$

値が大きいほど clean 側の答えが優勢です。3 種類の metric を考えます。

$$
m^{\text{clean}}      = m(z^{\text{clean}}), \quad
m^{\text{corrupt}}    = m(z^{\text{corrupt}}), \quad
m^{\text{patched}(k)} = m(z^{\text{patched}(k)})
$$

$m^{\text{clean}}$ が recovery の上限、$m^{\text{corrupt}}$ がパッチなしの下限です。

**Recovery**

$$
\text{recovery}(k)
= \frac{m^{\text{patched}(k)} - m^{\text{corrupt}}}
       {m^{\text{clean}} - m^{\text{corrupt}}}
$$

- $\text{recovery}(k) \approx 0$ ⇒ layer $k$ のパッチは効果なし（corrupt のまま）
- $\text{recovery}(k) \approx 1$ ⇒ layer $k$ のパッチで clean に完全回復

### hidden_states インデックスと patch site の対応

patching を行う層 $k$ は、Section 6 で見た `hidden_states[k]`（= $h_t^{(k)}$）のインデックスとそのまま対応します。

| $k$ | patch site | 対応するモジュール |
|---|---|---|
| $k = 0$ | embed_tokens の出力 | `model.model.embed_tokens` |
| $1 \le k \le K-1$ | layers[$k-1$] の出力 | `model.model.layers[k-1]` |
| $k = K$ | final RMSNorm の出力 | `model.model.norm` |

実装上は、対応するモジュールの forward hook で、出力テンソルの `[0, corrupt_pos, :]` を `patch_vec` (= `clean_hs[k][0, clean_pos, :]`) に置換します。


### 概念図

下のセルで、clean run / corrupt run / patched run の 3 経路を 1 枚の図にまとめます。  
layer $k$（図中の `L_k`）の位置で、clean run の活性が patched run に注入される様子を示しています。


In [ ]:
# Activation Patching の概念図 (schematic)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 4.5))

stages = ["emb", r"$L_0$", "...", r"$L_k$", "...", r"$L_{K-1}$", "norm", "lm_head"]
n = len(stages)
box_w, box_h = 0.10, 0.13
xs = [0.08 + i * 0.11 for i in range(n)]

# (label, y, color, output_text); label は改行で 2 行表記
rows = [
    ("clean run\n(clean prompt)",       0.73, "#bcdcff", r"$z^{\,\mathrm{clean}}$"),
    ("corrupt run\n(corrupt prompt)",   0.43, "#ffc4b8", r"$z^{\,\mathrm{corrupt}}$"),
    ("patched(k)\n(corrupt prompt)",    0.13, "#bbe6bb", r"$z^{\,\mathrm{patched}(k)}$"),
]

for label, y, color, output_text in rows:
    ax.text(0.06, y + box_h/2, label,
            ha="right", va="center", fontsize=10, fontweight="bold")
    for x, stage in zip(xs, stages):
        if label.startswith("patched") and stage == r"$L_k$":
            face = "#ffd84d"  # 注入された箱は強調
            ec = "red"
        else:
            face = color
            ec  = "black"
        ax.add_patch(mpatches.Rectangle((x, y), box_w, box_h,
                                        facecolor=face, edgecolor=ec, linewidth=1.2))
        ax.text(x + box_w/2, y + box_h/2, stage,
                ha="center", va="center", fontsize=9)
    for i in range(n - 1):
        ax.annotate("", xy=(xs[i+1], y + box_h/2),
                    xytext=(xs[i] + box_w, y + box_h/2),
                    arrowprops=dict(arrowstyle="->"))
    ax.text(xs[-1] + box_w + 0.015, y + box_h/2, output_text,
            ha="left", va="center", fontsize=11)

# clean L_k -> patched L_k の縦矢印
# 箱と重ならないよう、L_k 右端の外側に少しオフセット
k_idx = stages.index(r"$L_k$")
x_arrow = xs[k_idx] + box_w - 0.02
ax.annotate("",
            xy=(x_arrow, rows[2][1] + box_h),   # 矢頭: patched L_k 上端
            xytext=(x_arrow, rows[0][1]),       # 矢尾: clean L_k 下端
            arrowprops=dict(arrowstyle="->", color="red", lw=2.0))

# 注記 (2 行)、patched 行の上に配置して箱と重ならないようにする
ax.text(x_arrow + 0.010, rows[2][1] + box_h + 0.04,
        "patch: clean activation (at clean_pos)\n"
        r"$\rightarrow$ position corrupt_pos",
        ha="left", va="bottom", fontsize=9, color="red")

ax.set_xlim(0, 1.25)
ax.set_ylim(0, 1.0)
ax.axis("off")
ax.set_title("Activation Patching: inject clean activation at layer k into corrupt run",
             fontsize=12)
plt.tight_layout()
plt.show()


## 10. 補足：forward hook の仕組みと挿入場所

> **読み飛ばし可**：このセクションは、PyTorch の forward hook 機構と Transformers の実装に踏み込む補足です。Section 9 の `run_patch` と activation patching の結果を理解するだけなら、読み飛ばしてかまいません。なお、ここでの記述は内部実装に関する説明を含み、不正確な箇所が含まれている可能性があります。活性パッチングの本筋（Section 9 まで）と分けて扱い、誤りに気づいた場合は PyTorch / Transformers の公式ドキュメントやソースを優先してください。

`run_patch(k)` は内部で **forward hook** という PyTorch の仕組みを使い、ある module の `forward` が返した出力テンソルに介入しています。このセクションでは、その仕組みを次の 4 段階に分けて確認します。

1. **PyTorch の forward hook とは何か**：登録した関数が `forward` の直後に自動で呼ばれる仕掛け
2. **Qwen3 モデルの中で patching の target になりうる 3 種類のモジュール**：`embed_tokens` / `layers[k-1]` / `norm`
3. **`Qwen3Model.forward` のソースで hook が発動する位置を特定する**：どの行で hook が発動（最近は「発火」とも呼ぶ）するか
4. **各 target_module の `forward` が tensor 1 つを返すことの確認**：これにより `out.clone()` で素直に書き換えられる


### PyTorch の forward hook とは何か

`module.register_forward_hook(fn)` を呼ぶと、`module._forward_hooks` という辞書に `fn` が登録されます。  
その後 `module(input)` を呼んで `forward()` が実行され、出力が確定した直後に PyTorch が `_forward_hooks` を順に呼び出します。各 hook は `fn(module, inputs, outputs)` の形で呼ばれ、戻り値を返せば PyTorch はその戻り値を新しい出力として後続層に渡します。これが、activation patching で `out.clone()` を書き換えた tensor を返すと、後続の層が **書き換え後の値を見る** ことができる理由です。

下の cell は、`nn.Module.register_forward_hook` のソースを表示しています。内部の `RemovableHandle` の dict 操作などの詳細は読み飛ばしてかまいません。


In [ ]:
# register_forward_hook が何をするかを確認
# -> self._forward_hooks に hook 関数を登録する
# forward() が呼ばれるたびに _forward_hooks の中身が順番に実行される
import torch.nn as nn
print(inspect.getsource(nn.Module.register_forward_hook))

### Qwen3 モデルの target_module 3 種類

Section 9 の表で見たとおり、`run_patch(k)` の hook は patching 層 $k$ に応じて、次の 3 つのモジュールのどれか 1 つに登録されます。

- `k == 0`：`model.model.embed_tokens` （入力 token を embedding に変換する `nn.Embedding`）
- `1 <= k <= K-1`：`model.model.layers[k-1]` （Qwen3 の decoder block、`Qwen3DecoderLayer`）
- `k == K`：`model.model.norm` （最終 RMSNorm、`Qwen3RMSNorm`）

下の 2 つの cell では、まず 3 種類のモジュールの **型** を確認し、続いて実際のモジュールの中身（部分構造）を取り出して確認します。`layers[1], layers[2], ...` も `layers[0]` と同じ構造なので、代表として `layers[0]` のみ表示します。


In [ ]:
# run_patch で使う3種類の target_module の型を確認
print("embed_tokens:", type(model.model.embed_tokens))
print("layers[0]   :", type(model.model.layers[0]))
print("norm        :", type(model.model.norm))

In [ ]:
# patching の target になりうる 3 つのモジュールを実際に取り出して確認
print("=== model.model.embed_tokens ===")
print(model.model.embed_tokens)
print()
print("=== model.model.layers[0] === （layers[1..K-1] も同じ構造）")
print(model.model.layers[0])
print()
print("=== model.model.norm ===")
print(model.model.norm)


### `Qwen3Model.forward` のソースで hook が発動する位置を確認する

Qwen3 モデルの本体である `Qwen3Model.forward` のソースを表示します。注目するのは次の 3 行です。

- `inputs_embeds = self.embed_tokens(input_ids)` ← `k == 0` の hook が発動する箇所
- `hidden_states = decoder_layer(hidden_states, ...)`（for ループの中） ← `1 <= k <= K-1` の hook が発動する箇所
- `hidden_states = self.norm(hidden_states)` ← `k == K` の hook が発動する箇所

それぞれの `forward` 呼び出しが `return` した **直後** に hook が発火します。


In [ ]:
# Qwen3Model.forward — 計算順序を確認する
from transformers.models.qwen3.modeling_qwen3 import Qwen3Model
print(inspect.getsource(Qwen3Model.forward))

上記の `Qwen3Model.forward` のソースに対応する `run_patch(k)` の hook 登録先を表にまとめます。

| `k` | `Qwen3Model.forward` 内の該当行 | `target_module` |
|---|---|---|
| `k == 0` | `inputs_embeds = self.embed_tokens(input_ids)` | `model.model.embed_tokens` |
| `1 <= k <= K-1` | `hidden_states = decoder_layer(hidden_states, ...)`（for ループの $k' = k$ 回目、つまり出力が `hs[k]` になる回。`self.layers[k-1]` が適用される） | `model.model.layers[k-1]` |
| `k == K` | `hidden_states = self.norm(hidden_states)` | `model.model.norm` |

該当モジュールの `forward` が `return` した直後に hook 関数が呼ばれ、出力テンソルの `[0, corrupt_pos, :]` だけを `patch_vec` で置換します。


### 各 target_module の `forward` が tensor 1 つを返すことの確認

`run_patch` の hook 関数は、

```python
def hook(module, inp, out):
    out = out.clone()
    out[0, corrupt_pos, :] = patch_vec
    return out
```

のように `out` をそのまま tensor として扱い、`out[0, corrupt_pos, :] = ...` で 1 行だけ書き換えています。これが成立するのは、3 種類の target_module（`Qwen3DecoderLayer`、`nn.Embedding`、`Qwen3RMSNorm`）の `forward` が、いずれも **tuple ではなく単一の tensor** を返すからです。返り値が tuple なら、`out[0]` を取り出してから書き換える等の処理が必要になります。

以降の cell で、それぞれのモジュールの `forward` を確認します。まずは `Qwen3DecoderLayer.forward` の骨格を疑似コード化したものを示します。


```
residual = hidden_states
hidden_states = self.input_layernorm(hidden_states)
hidden_states = self.self_attn(...)          # Attention
hidden_states = residual + hidden_states     # 残差結合 (1)

residual = hidden_states
hidden_states = self.post_attention_layernorm(hidden_states)
hidden_states = self.mlp(hidden_states)      # MLP
hidden_states = residual + hidden_states     # 残差結合 (2)

return hidden_states   # ← hook はこの return の直後に呼ばれる
```

返り値が tuple でなく **tensor 1 つ** であることが見えます。これが、`out.clone()` で出力テンソルを直接書き換えられる理由です。下の cell に実際のソースを表示します。


In [ ]:
# decoder layer の forward を確認
# Qwen3DecoderLayer.forward（k=1..K-1 のとき）
from transformers.models.qwen3.modeling_qwen3 import Qwen3DecoderLayer, Qwen3RMSNorm
print(inspect.getsource(Qwen3DecoderLayer.forward))

次に `k == 0` のときの target である `nn.Embedding.forward` を確認します。これも返り値は単一の tensor です。


In [ ]:
# nn.Embedding.forward（k=0 のとき）
import torch.nn as nn
print(inspect.getsource(nn.Embedding.forward))

最後に `k == K` のときの target である `Qwen3RMSNorm.forward` を確認します。こちらも単一の tensor を返します。


In [ ]:
# Qwen3RMSNorm.forward（k=K のとき）
print(inspect.getsource(Qwen3RMSNorm.forward))

## 11. 全 layer スイープ

**数式記号と Python 変数の対応**

| 数式 | Python | 内容 |
|---|---|---|
| $z^{\text{clean}}$ | `clean_logits` | clean run の最終 logits |
| $z^{\text{corrupt}}$ | `corrupt_logits` | corrupt run の最終 logits |
| $z^{\text{patched}(k)}$ | `plogits`（`run_patch(k)` の戻り値） | layer $k$ をパッチした後の最終 logits |
| $m^{\text{clean}}$ | `clean_metric` | $m(z^{\text{clean}})$ |
| $m^{\text{corrupt}}$ | `corrupt_metric` | $m(z^{\text{corrupt}})$ |
| $m^{\text{patched}(k)}$ | `pm`（sweep loop 内） | $m(z^{\text{patched}(k)})$ |
| $\mathrm{recovery}(k)$ | `rec` | 上式どおり |

`metric(logits, clean_id, corrupt_id)`（前セルで定義） がこの $m(z) = z_{\text{clean\_id}} - z_{\text{corrupt\_id}}$ に対応します。


In [ ]:
# metric のベースライン（recovery 計算の分母）
clean_metric   = metric(clean_logits,   CLEAN_ANS_ID, CORRUPT_ANS_ID)
corrupt_metric = metric(corrupt_logits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
print(f"clean   metric = {clean_metric:.4f}")
print(f"corrupt metric = {corrupt_metric:.4f}")

# patching sweep
sweep_rows = []
print(f"Patching sweep: k = 0 ... {K}")
for k in range(K + 1):
    plogits = run_patch(k)
    pprobs = torch.softmax(plogits, dim=-1)
    pm  = metric(plogits, CLEAN_ANS_ID, CORRUPT_ANS_ID)
    rec = (pm - corrupt_metric) / (clean_metric - corrupt_metric)
    site = "embed" if k == 0 else (f"L{k-1:02d}" if k < K else "norm")
    sweep_rows.append({
        "k": k,
        "site": site,
        "p_clean_patched": pprobs[CLEAN_ANS_ID].item(),
        "p_corrupt_patched": pprobs[CORRUPT_ANS_ID].item(),
        "patched_metric": pm,
        "recovery": rec,
    })
    if k % 5 == 0 or k == K:
        print(f"  k={k:2d} ({site:6s}): recovery={rec:.4f}")

df_sweep = pd.DataFrame(sweep_rows)
out = outputs_dir / "nb02_patching_sweep.csv"
df_sweep.to_csv(out, index=False)
print(f"Saved: {out.name}")

display(df_sweep.set_index("k").style.set_caption("patching sweep").format({"recovery": "{:.4f}", "patched_metric": "{:.4f}", "p_clean_patched": "{:.4f}", "p_corrupt_patched": "{:.4f}"}))

### パッチ注入後の P(answer)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["p_clean_patched"],   label=f"P({CLEAN_ANSWER})  patched", marker="o", markersize=3)
ax.plot(df_sweep["k"], df_sweep["p_corrupt_patched"], label=f"P({CORRUPT_ANSWER}) patched", marker="s", markersize=3, linestyle="--")
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("probability")
ax.set_title(f"Activation Patching — P(answer) after patch\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.legend()
ax.set_xticks(range(0, K + 1, 4))
ax.grid(True, alpha=0.3)

# p_clean_patched が p_corrupt_patched を初めて上回る k に赤線を引く
cross = df_sweep[df_sweep["p_clean_patched"] > df_sweep["p_corrupt_patched"]]
if len(cross) > 0:
    k_cross = int(cross["k"].min())  # pyright: ignore[reportArgumentType]
    ax.axvline(k_cross, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_cross}: P({CLEAN_ANSWER.strip()}) > P({CORRUPT_ANSWER.strip()})")
    ax.legend()

plt.tight_layout()
out = outputs_dir / "nb02_patching_probs.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(cross) > 0:
    print(f"最初に 確率が入れ替わる layer: k={k_cross}")

### Recovery curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sweep["k"], df_sweep["recovery"], marker="o", markersize=4, color="steelblue", label="recovery")
ax.axhline(0.0, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("patch site k  (0=embed, K=norm)")
ax.set_ylabel("recovery")
ax.set_title(f"Activation Patching — Recovery by Layer\n({CLEAN_PROMPT}  →  {CORRUPT_PROMPT})")
ax.set_xticks(range(0, K + 1, 4))
ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3)

# 最初に recovery >= 0.5 になる k にマーカー
transition = df_sweep.loc[df_sweep["recovery"] >= 0.5, "k"]
if len(transition) > 0:
    k_transition = int(transition.min())
    ax.axvline(k_transition, color="tomato", linestyle=":", linewidth=1.5,
               label=f"k={k_transition}: recovery≥0.5")
    ax.legend()

plt.tight_layout()
out = outputs_dir / "nb02_recovery_curve.png"
plt.savefig(out, dpi=300)
plt.show()
print(f"Saved: {out.name}")
if len(transition) > 0:
    print(f"最初に recovery≥0.5 になる layer: k={k_transition}")


### 補足：Logit Lens と Activation Patching の比較

logit-difference metric で見ると、Logit Lens でも Activation Patching でも k=25 付近が転換点として現れる。
両手法は異なる操作だが、この題材では**同じ層を重要と示している**。

| 操作 | 内容 |
|------|------|
| **Logit Lens** | 層 k の残差ストリームを**その場で** lm_head（+norm）に通して読む |
| **Activation Patching** | 層 k の残差ストリームを差し替えたあと、**残りの層を通常通り forward** させる |

ただし、Activation Patching の recovery curve の方が変化がクリアに見える。
Logit Lens は「その時点の表現を直接読む」ため、後続の層による変換が反映されない。
Activation Patching は残りの層を通してから評価するため、残差ストリームの「影響力」をより直接的に測れる。

なお、確率（softmax）で見ると k=25 での変化は小さく見えるが、これは確率が後半層で急峻に飽和するためであり、
logit-difference metric を使うことで前半・中盤の変化が正確に読み取れる。

## 12. 特定 layer のパッチ後 top-k 比較

In [ ]:
# 特定 layer のパッチ後 logits を計算
plogits_24 = run_patch(24)
plogits_25 = run_patch(25)
plogits_K  = run_patch(K)

labels = [
    ("clean baseline",   clean_logits),
    ("corrupt baseline", corrupt_logits),
    ("patch k=24",       plogits_24),
    ("patch k=25",       plogits_25),
    (f"patch k=K={K}",   plogits_K),
]

for label, lgts in labels:
    rec_val = (metric(lgts, CLEAN_ANS_ID, CORRUPT_ANS_ID) - corrupt_metric) / (clean_metric - corrupt_metric)
    display(topk_table(lgts, k=5).style.set_caption(f"{label}  (recovery={rec_val:.4f})"))

# k=K サニティチェック: recovery=1.000 になるはず
recK = (metric(plogits_K, CLEAN_ANS_ID, CORRUPT_ANS_ID) - corrupt_metric) / (clean_metric - corrupt_metric)
print(f"k=K sanity: recovery = {recK:.6f}  (expected 1.000)")
if abs(recK - 1.0) > 1e-3:
    print("[warning] recovery != 1.000  (hook の実装を確認)")

## 13. まとめ

### 観察結果

| 手法 | 観察したこと |
|------|-------------|
| **Logit Lens** | 前半層（k≤24程度）では top-1 が意味のないトークン、後半層（k≈25以降）から正答 ` Tokyo` が上位に現れる |
| **Activation Patching** | k≤24 では recovery≈0、k=25 付近から急増、k≥34 では recovery≈1.0 |
| **k=K sanity check** | k=K（norm 後）のパッチで recovery=1.000 → hook の実装が正しいことを確認 |

### 解釈（この prompt pair とこの setup における観察）

- この題材・この last-token position・この metric では、**k=25 付近の残差ストリームを差し替えると最終出力が ` Tokyo` 側へ大きく変化した**。
- 少なくとも last-token position の残差ストリームを単独で差し替えるこの実験では、k≤24 の patch は最終出力をほとんど変えなかった。
- ただし、これを「知識が一般に後半層だけにある」と一般化してはいけない。あくまで **この prompt pair・この patching setup における観察**である。
- Logit Lens では ` Tokyo` の確率が明確に上がるのは k≈29 以降だが、Activation Patching では k=25 でも大きな recovery が得られた。これは両手法が**異なる操作**であることを反映している（詳細は Section 9 の注記を参照）。


### 参考文献・参考資料

- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems 30 (NeurIPS 2017).  
  URL: https://papers.nips.cc/paper/7181-attention-is-all-you-need  
  arXiv: https://arxiv.org/abs/1706.03762  
  Transformer architecture の基本論文。self-attention, feed-forward network, residual connection, layer normalization など、本ノートで扱う residual stream や後続 notebook で扱う Attention / FFN の背景となる。

- Alammar, J. (2018). *The Illustrated Transformer*.  
  URL: https://jalammar.github.io/illustrated-transformer/  
  Transformer の図解解説。原論文より直感的に読みやすく、embedding, self-attention, feed-forward network, encoder/decoder などの流れを図で確認するための補助資料として有用。

- nostalgebraist. (2020). *interpreting GPT: the logit lens*. LessWrong, August 31, 2020.  
  URL: https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens  
  中間層の hidden state に最終層と同じ読み出しを適用し、「その時点でのモデルの暫定的な次 token 予測」として読む logit lens の元アイデア。本ノートの logit lens の説明と実験に直接関係する。

- Meng, K., Bau, D., Andonian, A., & Belinkov, Y. (2022). *Locating and Editing Factual Associations in GPT*. Advances in Neural Information Processing Systems 35 (NeurIPS 2022).  
  arXiv: https://arxiv.org/abs/2202.05262  
  Project page: https://rome.baulab.info/  
  GPT 内部で factual prediction に関わる activations を causal intervention により調べ、ROME (Rank-One Model Editing) による model editing へ接続した研究。本ノートの residual stream patching は、この論文で用いられる Causal Tracing と同様に、clean/corrupt run の activation を差し替えて最終出力への影響を見る簡略化されたデモである。

- Heimersheim, S., & Nanda, N. (2024). *How to use and interpret activation patching*. arXiv:2404.15255.  
  URL: https://arxiv.org/abs/2404.15255  
  LessWrong version: https://www.lesswrong.com/posts/FhryNAFknqKAdDcYy/how-to-use-and-interpret-activation-patching  
  Activation patching の使い方と解釈上の注意を整理したチュートリアル。本ノートのような residual stream patching をどう解釈すべきか、metric の選び方や clean→corrupt / corrupt→clean の違いを考える際の補足資料。

- Nanda, N., & Bloom, J. (2022). *TransformerLens*. Software library.  
  URL: https://github.com/TransformerLensOrg/TransformerLens  
  Documentation: https://transformerlensorg.github.io/TransformerLens/  
  Citation: https://transformerlensorg.github.io/TransformerLens/content/citation.html  
  Transformer の内部 activations を取得・編集する mechanistic interpretability 用ライブラリ。logit lens や activation patching の実装・検証に広く使われる。本ノート本体では Hugging Face Transformers と自前 hook を使うが、事前調査スクリプトでは自前 logit lens 実装と TransformerLens 実装の一致を確認した。